<a href="https://colab.research.google.com/github/Kminhas21/ds2002-fa26/blob/main/2026_10_16_%E2%80%94_Walmart_Warm_Up_SKU_Mess_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [ ]:
import pandas as pd

PATH = '/content/06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [ ]:
pop = cat[cat['product_name'].str.lower().str.contains('pop', na=False)]
print(len(pop), 'Pop-Tart-looking rows')
pop

4 Pop-Tart-looking rows


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [ ]:
names = pop['product_name'].str.lower()
# 'pop' also catches popcorn etc., so require 'tart' to count as a Pop-Tart
is_tart = names.str.contains('tart')
straw = pop[is_tart & names.str.contains('straw')]
not_straw = pop[is_tart & ~names.str.contains('straw')]
print('strawberry Pop-Tart variants:')
display(straw[['sku', 'product_name']])
print('Pop-Tart rows that are NOT strawberry (keep separate):')
display(not_straw[['sku', 'product_name']])

strawberry Pop-Tart variants:


,sku,product_name
0,PT-12,pop-tarts strawberry
1,SKU#459812,Strawberry Pop-Tart
2,POPTART-STRAW,Pop-Tarts Strawberry
3,PT_STRAWBERRY,pop-tarts strawberry


Pop-Tart rows that are NOT strawberry (keep separate):


,sku,product_name


**Pop-Tart rows that are a different product:** None. All four Pop-Tart rows (`PT-12`, `SKU#459812`, `POPTART-STRAW`, `PT_STRAWBERRY`) are strawberry, so nothing has to stay separate on flavor. The one I'd keep an eye on is `POPTART-STRAW`, since it's listed as a 6 pack while the others are 4-packs.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [ ]:
CANONICAL = straw['sku'].iloc[0]   # pick one variant as the canonical SKU
sku_map = {s: CANONICAL for s in straw['sku'] if s != CANONICAL}
print(sku_map)

n_before = len(cat)   # snapshot for the Q6 check

# .replace() leaves unmapped SKUs alone; .map() would turn them into NaN
cat['canonical_sku'] = cat['sku'].replace(sku_map)
cat[['sku', 'canonical_sku', 'product_name']]

{'SKU#459812': 'PT-12', 'POPTART-STRAW': 'PT-12', 'PT_STRAWBERRY': 'PT-12'}


,sku,canonical_sku,product_name
0,PT-12,PT-12,pop-tarts strawberry
1,SKU#459812,PT-12,Strawberry Pop-Tart
2,POPTART-STRAW,PT-12,Pop-Tarts Strawberry
3,PT_STRAWBERRY,PT-12,pop-tarts strawberry
4,WTR-24,WTR-24,BOTTLED WATER 24PK
5,WATER24PK,WATER24PK,Bottled Water 24-Pack
6,WTR_24PK,WTR_24PK,Water - 24 Pack
7,BATT-100,BATT-100,BATTERIES AA 4PK
8,BAT-AA4,BAT-AA4,AA Batteries 4-Pack
9,BATT_AA_4PK,BATT_AA_4PK,AA Batteries 4-Pack


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [ ]:
print('before:', cat['category'].nunique(), 'labels')
print(cat['category'].value_counts(dropna=False))

cat['category_clean'] = cat['category'].str.strip().str.title()
print('\nafter strip + title:', cat['category_clean'].nunique(), 'labels')
print(cat['category_clean'].value_counts(dropna=False))

# casing can't fix synonyms, so map the leftover pairs explicitly
category_map = {
    'Breakfast Foods': 'Breakfast',
    'Emergency Supplies': 'Emergency',
    'Alcoholic Beverages': 'Alcohol',
    'Groceries': 'Grocery',
    'Baby Products': 'Baby',
}
cat['category_clean'] = cat['category_clean'].replace(category_map)
print('\nafter synonym map:', cat['category_clean'].nunique(), 'labels')
print(cat['category_clean'].value_counts())

before: 27 labels
category
Emergency              5
Emergency Supplies     3
EMERGENCY              3
ENTERTAINMENT          3
GROCERY                3
Breakfast              2
entertainment          2
household              2
ELECTRONICS            2
health                 2
Baby                   2
Electronics            1
ALCOHOL                1
Alcoholic Beverages    1
breakfast              1
Breakfast Foods        1
Groceries              1
Grocery                1
electronics            1
snacks                 1
grocery                1
HOUSEHOLD              1
Snacks                 1
Household              1
baby                   1
Baby Products          1
emergency              1
Name: count, dtype: int64

after strip + title: 15 labels
category_clean
Emergency              9
Entertainment          5
Grocery                5
Electronics            4
Household              4
Emergency Supplies     3
Breakfast              3
Baby                   3
Snacks                 2


Strip + title-case fixes the casing mess, but five pairs are still two labels for one thing: Breakfast/Breakfast Foods, Emergency/Emergency Supplies, Alcohol/Alcoholic Beverages, Grocery/Groceries, and Baby/Baby Products. Those need an explicit map on top, which takes it from 15 labels down to 10.

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [ ]:
import re

def name_key(name):
    words = re.sub(r'[^a-z0-9 ]', ' ', str(name).lower()).split()
    return ' '.join(sorted(words))

cat['name_key'] = cat['product_name'].apply(name_key)
groups = cat.groupby('name_key')['sku'].nunique()
candidates = groups[groups > 1].index
print(len(candidates), 'candidate groups')
cat[cat['name_key'].isin(candidates)].sort_values('name_key')[['name_key', 'sku', 'product_name', 'category']]

14 candidate groups


,name_key,sku,product_name,category
12,24 beer light pack,BEER-24,Light Beer (24-pack),Alcoholic Beverages
13,24 beer light pack,LTBEER24,Light Beer (24-pack),ALCOHOL
8,4 aa batteries pack,BAT-AA4,AA Batteries 4-Pack,Emergency Supplies
9,4 aa batteries pack,BATT_AA_4PK,AA Batteries 4-Pack,Emergency
30,aid first kit,FIRSTAID-KIT,First Aid Kit,health
29,aid first kit,FAID-01,First-Aid Kit,health
15,board game monopoly,MONOPOLY-01,Monopoly Board Game,entertainment
14,board game monopoly,BG-MONO,Monopoly Board Game,ENTERTAINMENT
16,board game monopoly,GAME-MONO,Board Game - Monopoly,ENTERTAINMENT
24,bread white,WHITEBREAD,WHITE BREAD,grocery


This surfaces 14 candidate groups, including Monopoly, AA batteries, light beer, and flashlights. It still misses anything where the wording or pack notation differs, like all three bottled waters (`24PK` vs `24-Pack`) and `Strawberry Pop-Tart` vs `Pop-Tarts`, so it's a starting list, not the final mapping.

### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [ ]:
assert cat['canonical_sku'].notna().all(), 'mapping produced nulls'
assert len(cat) == n_before, 'row count changed'
# every original SKU is accounted for: either kept or mapped to the canonical one
assert set(cat['canonical_sku']) <= set(cat['sku']) | {CANONICAL}
print(f'mapping is safe: {n_before} rows, 0 nulls, '
      f"{cat['sku'].nunique()} -> {cat['canonical_sku'].nunique()} distinct SKUs")

mapping is safe: 45 rows, 0 nulls, 45 -> 42 distinct SKUs


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

**a)** Bottled water (`WTR-24`, `WATER24PK`, `WTR_24PK`) and AA batteries (`BATT-100`, `BAT-AA4`, `BATT_AA_4PK`) are both split across three SKUs. I'd group each on product + size (24-pack water, AA 4-pack) and map the variants to one canonical SKU the same way I did for Pop-Tarts.

**b)** It turns into a business call once the rows *might* be different real products and the names can't settle it. I'm honestly not sure `STREAM-01` (Roku Streaming Stick) and `ROKU-STD` (Streaming Device) are the same item, since one is brand-specific and the other is generic, so I'd ask the electronics category/merchandising manager.